In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# Definindo os caminhos - notebook em scripts/Treinamento DNN/
caminho_pasta_tratado = '../../dataset tratado/lycos-cicids2017/'

nome_dados_treinamento = 'Redução de Dimensionalidade/lycos_cicids2017_treinamento_reduzidos.csv'
nome_dados_teste = 'Redução de Dimensionalidade/lycos_cicids2017_teste_reduzidos.csv'

In [3]:
print("Carregando os datasets com redução de dimensionalidade...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")
display(df_treino.head())
display(df_teste.head())

Carregando os datasets com redução de dimensionalidade...
Dados carregados! Treino: (1286248, 52) | Teste: (551250, 52)


,pkt_len_std,pkt_len_max,pkt_len_var,flag_rst,bwd_pkt_len_max,fwd_pkt_len_mean,fwd_pkt_len_max,fwd_pkt_len_tot,bwd_pkt_len_mean,bwd_pkt_len_std,...,bwd_iat_tot,bwd_pkt_len_tot,fwd_iat_tot,bwd_iat_mean,src_port,active_max,bwd_bulk_bytes_mean,iat_min,iat_std,Label
0,0.007094,0.003948,0.000050,0.0,0.005018,0.007743,0.001853,0.000015,0.022422,0.000000,...,0.000000e+00,1.495151e-07,0.000000e+00,0.000000e+00,0.951110,0.000000,0.000000,8.763038e-04,0.000000,benign
1,0.002960,0.001249,0.000009,0.0,0.000000,0.001796,0.001249,0.000010,0.000000,0.000000,...,0.000000e+00,0.000000e+00,2.549179e-02,0.000000e+00,0.766598,0.000000,0.000000,8.985502e-07,0.018075,benign
2,0.066308,0.057131,0.004397,0.0,0.072606,0.010268,0.023852,0.000390,0.061572,0.068549,...,9.793137e-01,7.390316e-06,9.796187e-01,5.761774e-02,0.814466,0.004015,0.000004,2.173912e-07,0.158063,benign
3,0.025285,0.010475,0.000639,0.0,0.013313,0.005555,0.001330,0.000021,0.059487,0.000000,...,3.333334e-08,7.933453e-07,2.500000e-08,3.333973e-08,0.074983,0.000000,0.000000,2.463767e-07,0.000002,benign
4,0.204189,0.175020,0.041693,0.0,0.222427,0.004675,0.020830,0.000551,0.387286,0.131377,...,7.001895e-02,1.936876e-04,2.984716e-02,9.463836e-04,0.847898,0.000000,0.000387,2.028984e-07,0.005670,benign


,pkt_len_std,pkt_len_max,pkt_len_var,flag_rst,bwd_pkt_len_max,fwd_pkt_len_mean,fwd_pkt_len_max,fwd_pkt_len_tot,bwd_pkt_len_mean,bwd_pkt_len_std,...,bwd_iat_tot,bwd_pkt_len_tot,fwd_iat_tot,bwd_iat_mean,src_port,active_max,bwd_bulk_bytes_mean,iat_min,iat_std,Label
0,0.015594,0.007131,0.000243,0.0,0.009063,0.006228,0.001491,0.000024,0.040497,0.0,...,2.500000e-08,5.400850e-07,2.500000e-08,2.500480e-08,0.918364,0.0,0.0,2.463767e-07,1.170775e-06,benign
1,0.005347,0.003828,0.000029,0.0,0.004864,0.007912,0.001894,0.000030,0.021736,0.0,...,2.500000e-08,2.898762e-07,2.500000e-08,2.500480e-08,0.838499,0.0,0.0,2.463767e-07,1.068673e-06,benign
2,0.019159,0.008461,0.000367,0.0,0.010753,0.006397,0.001531,0.000024,0.048047,0.0,...,4.000000e-07,6.407789e-07,4.083333e-07,4.000768e-07,0.748730,0.0,0.0,8.985502e-07,7.862114e-07,benign
3,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.556161,0.0,0.0,1.173912e-06,0.000000e+00,portscan
4,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.754528,0.0,0.0,1.942028e-06,0.000000e+00,portscan


In [4]:
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

In [5]:
print("Codificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
y_teste_num = encoder.transform(y_teste)
num_classes = len(encoder.classes_)

Codificando as labels para a Rede Neural...


In [6]:
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [7]:
print("\nIniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...
Epoch 1/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9856 - loss: 0.0649 - val_accuracy: 0.9956 - val_loss: 0.0164
Epoch 2/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9958 - loss: 0.0162 - val_accuracy: 0.9967 - val_loss: 0.0112
Epoch 3/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9965 - loss: 0.0122 - val_accuracy: 0.9971 - val_loss: 0.0098
Epoch 4/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9968 - loss: 0.0106 - val_accuracy: 0.9972 - val_loss: 0.0089
Epoch 5/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9970 - loss: 0.0097 - val_accuracy: 0.9971 - val_loss: 0.0086
Epoch 6/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9972 - loss: 0.0089 - val_accuracy: 0.9974 - val_loss: 0.0076
Epoch 7/10
4522/4522 ━━━━━━━━━━━━━━━━━━━━ 5s 1ms/step - accuracy: 0.9972 - loss: 0.0086 - val_accuracy: 0.9975 - val_loss: 0.0074
Ep

In [8]:
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

print("\n=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
17227/17227 ━━━━━━━━━━━━━━━━━━━━ 8s 442us/step
Classificação concluída! Tempo de previsão: 11.7958 segundos.

=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===


,Pred_benign,Pred_bot,Pred_ddos,Pred_dos_goldeneye,Pred_dos_hulk,Pred_dos_slowhttptest,Pred_dos_slowloris,Pred_ftp_patator,Pred_heartbleed,Pred_portscan,Pred_ssh_patator,Pred_webattack_bruteforce,Pred_webattack_sql_injection,Pred_webattack_xss
Real_benign,418149,6,0,249,1,127,79,1,0,24,1,2,0,0
Real_bot,2,217,0,0,0,0,0,0,0,0,0,0,0,0
Real_ddos,9,0,28587,0,0,0,0,0,0,0,0,0,0,0
Real_dos_goldeneye,16,0,0,2079,0,1,0,0,0,0,0,0,0,0
Real_dos_hulk,4,0,0,0,47932,3,0,0,0,3,0,0,0,0
Real_dos_slowhttptest,31,0,0,1,0,1388,3,0,0,0,0,0,0,0
Real_dos_slowloris,5,0,0,0,0,22,1739,0,0,9,0,0,0,0
Real_ftp_patator,3,0,0,0,0,0,0,1178,0,0,0,0,0,0
Real_heartbleed,1,0,0,0,0,0,0,0,3,0,0,0,0,0
Real_portscan,21,0,0,0,0,0,0,0,0,47886,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                         precision    recall  f1-score   support

                 benign       1.00      1.00      1.00    418639
                    bot       0.97      0.99      0.98       219
                   ddos       1.00      1.00      1.00     28596
          dos_goldeneye       0.89      0.99      0.94      2096
               dos_hulk       1.00      1.00      1.00     47942
       dos_slowhttptest       0.90      0.98      0.94      1423
          dos_slowloris       0.95      0.98      0.97      1775
            ftp_patator       1.00      1.00      1.00      1181
             heartbleed       1.00      0.75      0.86         4
               portscan       1.00      1.00      1.00     47907
            ssh_patator       1.00      0.98      0.99       875
   webattack_bruteforce       0.94      0.08      0.15       397
webattack_sql_injection       0.00      0.00      0.00         3
          webattack_xss      

In [9]:
CLASS_ALIASES_LATEX = {'benign': 'BENIGN', 'bot': 'Bot', 'ddos': 'DDoS', 'dos_goldeneye': 'DoS GE', 'dos_hulk': 'Hulk', 'dos_slowhttptest': 'Slowhttp', 'dos_slowloris': 'Slowloris', 'ftp_patator': 'FTP', 'heartbleed': 'Heart', 'portscan': 'PortScan', 'ssh_patator': 'SSH', 'webattack_bruteforce': 'Brute', 'webattack_sql_injection': 'SQL', 'webattack_xss': 'XSS'}


def escape_latex(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return rf"\ok{{{value}}}"
    if value != 0:
        return rf"\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((rf"Real\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \scriptsize",
        r"    \resizebox{\linewidth}{!}{",
        rf"        \begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        r"            \hline",
        format_row("", headers),
        r"            \hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        r"            \hline",
        r"        \end{tabular}",
        r"    }",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        [r"\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        [r"\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        [r"\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + r" \\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \small",
        r"    \begin{tabular}{lrrrr}",
        r"        \hline",
        format_row(headers),
        r"        \hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        r"        \hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        r"        \hline",
        r"    \end{tabular}",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_matriz_confusao = make_latex_confusion_matrix(
    cm,
    labels_latex,
    "Matriz de Confusão — DNN com Redução de Dimensionalidade (LycoS-IDS2017)",
    "table:dnn_lycos_reducao_mc",
)
latex_relatorio_metricas = make_latex_metrics_report(
    y_true_latex,
    y_pred_latex,
    labels_latex,
    "Relatório de Métricas — DNN com Redução de Dimensionalidade (LycoS-IDS2017)",
    "table:dnn_lycos_reducao_metricas",
)

print(latex_matriz_confusao)
print("\n")
print(latex_relatorio_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrr}
            \hline
                            & BENIGN      & Bot      & DDoS       & DoS GE    & Hulk       & Slowhttp  & Slowloris & FTP       & Heart  & PortScan   & SSH      & Brute   & SQL    & XSS    \\
            \hline
            Real\_BENIGN    & \ok{418149} & \err{6}  & 0          & \err{249} & \err{1}    & \err{127} & \err{79}  & \err{1}   & 0      & \err{24}   & \err{1}  & \err{2} & 0      & 0      \\
            Real\_Bot       & \err{2}     & \ok{217} & 0          & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0       & 0      & 0      \\
            Real\_DDoS      & \err{9}     & 0        & \ok{28587} & 0         & 0          & 0         & 0         & 0         & 0      & 0          & 0        & 0       & 0      & 0      \\
            Real\_DoS GE    & \err{16}    & 0        & 0          & \ok{2079}